# 📚 Phase 1 — Data Preparation
## Audible Insights: Intelligent Book Recommendation System
---
### What we do in this notebook:
- **Part A** → Load & Explore both datasets
- **Part B** → Merge the two datasets into one
- **Part C** → Clean the merged data
- **Save** → Export cleaned_data.csv for all future phases


In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


---
## 📂 Part A — Load & Explore the Datasets

In [6]:
# Step 1: Load both CSV files
df1 = pd.read_csv("../data/Audible_Catlog.csv")
df2 = pd.read_csv("../data/Audible_Catlog_Advanced_Features.csv")

print(f"Dataset 1 shape : {df1.shape}  →  {df1.shape[0]} books, {df1.shape[1]} columns")
print(f"Dataset 2 shape : {df2.shape}  →  {df2.shape[0]} books, {df2.shape[1]} columns")

Dataset 1 shape : (6368, 5)  →  6368 books, 5 columns
Dataset 2 shape : (4464, 8)  →  4464 books, 8 columns


In [7]:
# Step 2: Explore Dataset 1
print("=== Dataset 1 - Columns ===")
print(df1.columns.tolist())
print()
print("=== First 3 rows ===")
df1.head(3)

=== Dataset 1 - Columns ===
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price']

=== First 3 rows ===


,Book Name,Author,Rating,Number of Reviews,Price
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,313.0,10080.0
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3658.0,615.0
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20174.0,10378.0


In [8]:
# Missing values and duplicates in Dataset 1
print("=== Missing Values ===")
print(df1.isnull().sum())
print()
print(f"Duplicate rows: {df1.duplicated().sum()}")
print()
print("=== Basic Statistics ===")
df1.describe()

=== Missing Values ===
Book Name              0
Author                 0
Rating                 0
Number of Reviews    631
Price                  3
dtype: int64

Duplicate rows: 929

=== Basic Statistics ===


,Rating,Number of Reviews,Price
count,6368.000000,5737.000000,6365.000000
mean,3.913709,902.786822,923.212726
std,1.663320,2454.003227,1551.750993
min,-1.000000,1.000000,0.000000
25%,4.200000,64.000000,501.000000
50%,4.500000,231.000000,680.000000
75%,4.600000,746.000000,888.000000
max,5.000000,70077.000000,18290.000000


In [9]:
# Step 3: Explore Dataset 2
print("=== Dataset 2 - Columns ===")
print(df2.columns.tolist())
print()
print("=== First 3 rows ===")
df2.head(3)

=== Dataset 2 - Columns ===
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre']

=== First 3 rows ===


,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,371.0,10080,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3682.0,615,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20306.0,10378,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top..."


In [10]:
# Missing values and duplicates in Dataset 2
print("=== Missing Values ===")
print(df2.isnull().sum())
print()
print(f"Duplicate rows: {df2.duplicated().sum()}")

=== Missing Values ===
Book Name              0
Author                 0
Rating                 0
Number of Reviews    421
Price                  0
Description            6
Listening Time         0
Ranks and Genre        0
dtype: int64

Duplicate rows: 168


---
## 🔗 Part B — Merge Both Datasets
We merge on **Book Name** + **Author** since these are common to both files.
Dataset 2 has richer info (Description, Genre, Listening Time) so we use it as the base.

In [11]:
# Step 4: Merge datasets
df = pd.merge(df2, df1[["Book Name", "Author", "Number of Reviews", "Price"]],
              on=["Book Name", "Author"],
              how="left",
              suffixes=("_adv", "_cat"))

# Resolve duplicate columns — prefer advanced features, fallback to catalog
df["Number of Reviews"] = df["Number of Reviews_adv"].combine_first(df["Number of Reviews_cat"])
df["Price"]             = df["Price_adv"].combine_first(df["Price_cat"])

# Drop redundant split columns
df.drop(columns=["Number of Reviews_adv", "Number of Reviews_cat",
                 "Price_adv", "Price_cat"], inplace=True, errors="ignore")

print(f"✅ Merged shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

✅ Merged shape: (5000, 8)
Columns: ['Book Name', 'Author', 'Rating', 'Description', 'Listening Time', 'Ranks and Genre', 'Number of Reviews', 'Price']


,Book Name,Author,Rating,Description,Listening Time,Ranks and Genre,Number of Reviews,Price
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top...",371.0,10080
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top...",3682.0,615
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top...",20306.0,10378


---
## 🧹 Part C — Data Cleaning
Fix all issues found during exploration: nulls, duplicates, wrong types, inconsistent text.

In [12]:
# Step 5a: Drop rows where Book Name is missing
before = len(df)
df.dropna(subset=["Book Name"], inplace=True)
print(f"Dropped {before - len(df)} rows with missing Book Name")

Dropped 0 rows with missing Book Name


In [13]:
# Step 5b: Fill missing text fields
df["Description"] = df["Description"].fillna("")
print("✅ Filled missing Descriptions with empty string")

✅ Filled missing Descriptions with empty string


In [14]:
# Step 5c-5e: Fill missing numeric fields with median
df["Rating"]            = df["Rating"].fillna(df["Rating"].median())
df["Number of Reviews"] = df["Number of Reviews"].fillna(df["Number of Reviews"].median())
df["Price"]             = df["Price"].fillna(df["Price"].median())

print(f"✅ Rating median fill    : {df['Rating'].median()}")
print(f"✅ Reviews median fill   : {df['Number of Reviews'].median()}")
print(f"✅ Price median fill     : {df['Price'].median()}")

✅ Rating median fill    : 4.5
✅ Reviews median fill   : 254.0
✅ Price median fill     : 683.0


In [15]:
# Step 5f: Remove duplicate books
before = len(df)
df.drop_duplicates(subset=["Book Name", "Author"], inplace=True)
print(f"✅ Removed {before - len(df)} duplicate books")
print(f"   Remaining: {len(df)} books")

✅ Removed 987 duplicate books
   Remaining: 4013 books


In [16]:
# Step 5g: Standardize text casing
df["Book Name"] = df["Book Name"].str.strip().str.title()
df["Author"]    = df["Author"].str.strip().str.title()
print("✅ Standardized Book Name and Author")

✅ Standardized Book Name and Author


In [17]:
# Step 5h: Ensure correct data types
df["Rating"]            = pd.to_numeric(df["Rating"],            errors="coerce")
df["Number of Reviews"] = pd.to_numeric(df["Number of Reviews"], errors="coerce")
df["Price"]             = pd.to_numeric(df["Price"],             errors="coerce")
print("✅ Converted numeric columns")
print(df.dtypes)

✅ Converted numeric columns
Book Name             object
Author                object
Rating               float64
Description           object
Listening Time        object
Ranks and Genre       object
Number of Reviews    float64
Price                  int64
dtype: object


In [18]:
# Step 5i: Extract Genre from Ranks and Genre column
def extract_genre(text):
    if pd.isna(text) or text == "": return "Unknown"
    parts = str(text).split(",")
    for part in parts:
        part = part.strip()
        if "in " in part:
            genre = part.split("in ")[-1].strip()
            if "Audible Audiobooks" not in genre and "See Top" not in genre:
                return genre
    return "Unknown"

df["Genre"] = df["Ranks and Genre"].apply(extract_genre)
print("✅ Extracted Genre")
print("Top 10 Genres:")
print(df["Genre"].value_counts().head(10))

✅ Extracted Genre
Top 10 Genres:
Genre
Unknown                                   2029
Personal Success                            94
Classic Literature                          45
Literary Fiction                            29
Leadership                                  27
Romance (Books)                             22
Business Careers                            18
Business Sales                              18
Textbooks & Study Guides                    18
Business Motivation & Self-Improvement      17
Name: count, dtype: int64


In [19]:
# Step 5j: Parse Listening Time → minutes
def parse_listening_time(text):
    if pd.isna(text) or text == "": return np.nan
    hours, mins = 0, 0
    text = str(text).lower()
    if "hour" in text:
        try: hours = int(text.split("hour")[0].strip().split()[-1])
        except: pass
    if "minute" in text:
        try: mins = int(text.split("minute")[0].strip().split()[-1])
        except: pass
    return hours * 60 + mins

df["Listening Time (mins)"] = df["Listening Time"].apply(parse_listening_time)
print("✅ Parsed Listening Time to minutes")
df[["Book Name", "Listening Time", "Listening Time (mins)"]].head(5)

✅ Parsed Listening Time to minutes


,Book Name,Listening Time,Listening Time (mins)
0,Think Like A Monk: The Secret Of How To Harnes...,10 hours and 54 minutes,654
1,Ikigai: The Japanese Secret To A Long And Happ...,3 hours and 23 minutes,203
2,The Subtle Art Of Not Giving A F*Ck: A Counter...,5 hours and 17 minutes,317
3,Atomic Habits: An Easy And Proven Way To Build...,5 hours and 35 minutes,335
4,Life'S Amazing Secrets: How To Find Balance An...,6 hours and 25 minutes,385


---
## 📊 Final Dataset Summary

In [20]:
# Step 6: Final check
print("=" * 50)
print(f"  Final Shape     : {df.shape}")
print(f"  Columns         : {df.columns.tolist()}")
print()
print("Missing Values after cleaning:")
print(df.isnull().sum())
print()
print(f"Rating range    : {df["Rating"].min()} – {df["Rating"].max()}")
print(f"Unique Authors  : {df["Author"].nunique()}")
print(f"Unique Genres   : {df["Genre"].nunique()}")
print("=" * 50)

  Final Shape     : (4013, 10)
  Columns         : ['Book Name', 'Author', 'Rating', 'Description', 'Listening Time', 'Ranks and Genre', 'Number of Reviews', 'Price', 'Genre', 'Listening Time (mins)']

Missing Values after cleaning:
Book Name                0
Author                   0
Rating                   0
Description              0
Listening Time           0
Ranks and Genre          0
Number of Reviews        0
Price                    0
Genre                    0
Listening Time (mins)    0
dtype: int64

Rating range    : -1.0 – 5.0
Unique Authors  : 2694
Unique Genres   : 633


In [21]:
# Preview final cleaned dataset
df[["Book Name", "Author", "Rating", "Number of Reviews", "Genre", "Listening Time (mins)"]].head(10)

,Book Name,Author,Rating,Number of Reviews,Genre,Listening Time (mins)
0,Think Like A Monk: The Secret Of How To Harnes...,Jay Shetty,4.9,371.0,Personal Success,654
1,Ikigai: The Japanese Secret To A Long And Happ...,Héctor García,4.6,3682.0,Self-Esteem,203
2,The Subtle Art Of Not Giving A F*Ck: A Counter...,Mark Manson,4.4,20306.0,Personal Success,317
3,Atomic Habits: An Easy And Proven Way To Build...,James Clear,4.6,4678.0,Stress Management,335
4,Life'S Amazing Secrets: How To Find Balance An...,Gaur Gopal Das,4.6,4308.0,Literary Essays,385
5,Sapiens,Yuval Noah Harari,4.6,20163.0,History of Civilization,918
7,Extraordinary Leadership,Robin Sharma,4.1,179.0,Leadership,66
8,The Intelligent Investor Rev Ed.,Benjamin Graham,4.4,9339.0,Personal Finance,1068
9,Rich Dad Poor Dad: What The Rich Teach Their K...,Robert T. Kiyosaki,4.5,22123.0,Personal Finance,369
10,The 5Am Club: Own Your Morning. Elevate Your L...,Robin Sharma,4.4,5116.0,Sleep Disorders,664


---
## 💾 Save Cleaned Dataset

In [22]:
# Step 7: Save to outputs folder
df.to_csv("../outputs/cleaned_data.csv", index=False)
print("✅ Saved: outputs/cleaned_data.csv")
print(f"📦 {df.shape[0]} books ready for Phase 2 (EDA)!")

✅ Saved: outputs/cleaned_data.csv
📦 4013 books ready for Phase 2 (EDA)!
